# Notebook 06 — Model 2: LightGBM

## Objective

Build a strong LightGBM multiclass classifier for SME Financial Health prediction, using leakage-safe categorical encoding and the engineered features developed in Notebook 05.

This notebook has two goals:

1. Produce a competitive standalone LightGBM model
2. Create a complementary model to CatBoost for later ensembling

Unlike CatBoost, LightGBM does not natively handle raw string categoricals in the same way, so categorical encoding must be handled carefully inside each cross-validation fold to avoid leakage.

In [2]:
# Imports and setup

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.base import clone

import lightgbm as lgb

from src.preprocessing import PreprocessConfig, preprocess_train_test

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

RANDOM_STATE = 42
TRAIN_PATH = "D:\\dataorg-financial-health-prediction-challenge20251204-19827-m2tn1n\\Train.csv"
TEST_PATH  = "D:\\dataorg-financial-health-prediction-challenge20251204-19827-m2tn1n\\Test.csv"

In [3]:
# Load raw data and apply preprocessing pipeline

train_raw = pd.read_csv(TRAIN_PATH)
test_raw  = pd.read_csv(TEST_PATH)

cfg = PreprocessConfig(id_col="ID", target_col="Target")

# LightGBM-friendly preprocessing: categorical missing -> "missing"
train, test = preprocess_train_test(train_raw, test_raw, cfg, for_model="lightgbm")

TARGET = cfg.target_col
ID = cfg.id_col

y = train[TARGET].copy()
X = train.drop(columns=[TARGET]).copy()

print("train shape:", train.shape)
print("test shape :", test.shape)

train shape: (9618, 47)
test shape : (2405, 46)


In [4]:
# Redefine the engineered features from Notebook 05

def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    def yn_to01(s):
        s = s.astype("string").str.lower().fillna("missing")
        return s.isin(["yes", "y", "1", "true"]).astype(int)

    # Access score
    access_cols = [c for c in [
        "has_credit_card",
        "has_debit_card",
        "has_loan_account",
        "has_internet_banking",
        "has_mobile_money",
    ] if c in df.columns]

    for c in access_cols:
        df[f"{c}__01"] = yn_to01(df[c])

    if access_cols:
        df["access_score"] = df[[f"{c}__01" for c in access_cols]].sum(axis=1)

    # Insurance score
    insurance_cols = [c for c in [
        "funeral_insurance",
        "medical_insurance",
        "motor_vehicle_insurance",
        "has_insurance",
    ] if c in df.columns]

    for c in insurance_cols:
        df[f"{c}__01"] = yn_to01(df[c])

    if insurance_cols:
        df["insurance_score"] = df[[f"{c}__01" for c in insurance_cols]].sum(axis=1)

    # Stress score
    stress_cols = [c for c in [
        "current_problem_cash_flow",
        "attitude_worried_shutdown",
        "problem_sourcing_money",
    ] if c in df.columns]

    for c in stress_cols:
        df[f"{c}__01"] = yn_to01(df[c])

    if stress_cols:
        df["stress_score"] = df[[f"{c}__01" for c in stress_cols]].sum(axis=1)

    # Formalization score
    formal_cols = [c for c in [
        "keeps_financial_records",
        "compliance_income_tax",
    ] if c in df.columns]

    for c in formal_cols:
        df[f"{c}__01"] = yn_to01(df[c])

    if formal_cols:
        df["formalization_score"] = df[[f"{c}__01" for c in formal_cols]].sum(axis=1)

    # Numeric ratios
    if "business_turnover" in df.columns and "business_expenses" in df.columns:
        turn = pd.to_numeric(df["business_turnover"], errors="coerce")
        exp  = pd.to_numeric(df["business_expenses"], errors="coerce")

        df["turnover_minus_expenses"] = turn - exp
        df["turnover_to_expenses"] = turn / exp.replace(0, np.nan)
        df["turnover_to_expenses"] = df["turnover_to_expenses"].replace([np.inf, -np.inf], np.nan)

    # Age bucket
    if "business_age_total_months" in df.columns:
        years = pd.to_numeric(df["business_age_total_months"], errors="coerce") / 12.0
        age_bucket = pd.cut(
            years,
            bins=[-np.inf, 0.5, 2, 5, 10, np.inf],
            labels=["<6m", "0.5-2y", "2-5y", "5-10y", "10y+"]
        )
        df["age_bucket"] = age_bucket.astype("string").fillna("missing")

    return df

X_fe = add_features(X).replace({pd.NA: np.nan})
test_fe = add_features(test).replace({pd.NA: np.nan})

print("X_fe shape:", X_fe.shape)
print("test_fe shape:", test_fe.shape)

X_fe shape: (9618, 67)
test_fe shape: (2405, 67)
